In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import roc_auc_score
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

print("Libraries loaded")

Libraries loaded


In [2]:
df = pd.read_csv('loan.csv', low_memory=False)
print(f"Raw shape: {df.shape}")
print(f"\nLoan status counts:")
print(df['loan_status'].value_counts())

Raw shape: (2260668, 145)

Loan status counts:
loan_status
Fully Paid                                             1041952
Current                                                 919695
Charged Off                                             261655
Late (31-120 days)                                       21897
In Grace Period                                           8952
Late (16-30 days)                                         3737
Does not meet the credit policy. Status:Fully Paid        1988
Does not meet the credit policy. Status:Charged Off        761
Default                                                     31
Name: count, dtype: int64


In [3]:
keep_status = ['Fully Paid', 'Charged Off', 'Default',
               'Late (31-120 days)', 'Late (16-30 days)']
df = df[df['loan_status'].isin(keep_status)].copy()

df['target'] = df['loan_status'].apply(lambda x: 0 if x == 'Fully Paid' else 1)

print(f"Shape after filtering: {df.shape}")
print(f"Class balance: {df['target'].mean()*100:.2f}% default")

Shape after filtering: (1329272, 146)
Class balance: 21.61% default


In [4]:
df['issue_d'] = pd.to_datetime(df['issue_d'], format='%b-%Y')
df['year'] = df['issue_d'].dt.year

print("Loans per year:")
print(df.groupby('year')['target'].agg(['count', 'mean']).round(3))

Loans per year:
       count   mean
year               
2007     251  0.179
2008    1562  0.158
2009    4716  0.126
2010   11536  0.129
2011   21721  0.152
2012   53367  0.162
2013  134803  0.156
2014  221902  0.187
2015  375220  0.205
2016  280472  0.258
2017  168699  0.274
2018   55023  0.269


In [5]:
train = df[df['year'] <= 2014].copy()
val   = df[df['year'] == 2015].copy()
test  = df[df['year'] >= 2016].copy()

print(f"Train: {len(train):,} rows | Default rate: {train['target'].mean():.3f}")
print(f"Val:   {len(val):,} rows | Default rate: {val['target'].mean():.3f}")
print(f"Test:  {len(test):,} rows | Default rate: {test['target'].mean():.3f}")

Train: 449,858 rows | Default rate: 0.171
Val:   375,220 rows | Default rate: 0.205
Test:  504,194 rows | Default rate: 0.265


In [6]:
# Drop cols with >40% missing (computed on full df)
missing_ratio = df.isnull().mean()
high_missing = missing_ratio[missing_ratio > 0.4].index.tolist()
print(f"Dropping {len(high_missing)} columns with >40% missing")

# Drop ID, outcome, and non-predictive columns
drop_always = ['id', 'member_id', 'loan_status', 'url', 'desc',
               'title', 'zip_code', 'issue_d', 'year']

cols_to_drop = list(set(high_missing + drop_always))
cols_to_drop = [c for c in cols_to_drop if c in df.columns]

df    = df.drop(columns=cols_to_drop)
train = train.drop(columns=[c for c in cols_to_drop if c in train.columns])
val   = val.drop(columns=[c for c in cols_to_drop if c in val.columns])
test  = test.drop(columns=[c for c in cols_to_drop if c in test.columns])

print(f"Remaining columns: {df.shape[1]}  (including target)")
print(f"Missing in df: {df.isnull().sum().sum():,}")

Dropping 58 columns with >40% missing
Remaining columns: 84  (including target)
Missing in df: 2,524,971


In [7]:
# Drop emp_title — free text, not useful for modelling
for df_ in [df, train, val, test]:
    df_.drop(columns=['emp_title'], inplace=True, errors='ignore')

# Define column types on cleaned df (excluding target)
num_cols = [c for c in df.select_dtypes(include='number').columns if c != 'target']
cat_cols = df.select_dtypes(include='object').columns.tolist()

print(f"Numerical features: {len(num_cols)}")
print(f"Categorical features: {len(cat_cols)}")
print(f"\nCategorical columns: {cat_cols}")
print(f"\nMissing values in df: {df.isnull().sum().sum():,}")

Numerical features: 65
Categorical features: 17

Categorical columns: ['term', 'grade', 'sub_grade', 'emp_length', 'home_ownership', 'verification_status', 'pymnt_plan', 'purpose', 'addr_state', 'earliest_cr_line', 'initial_list_status', 'last_pymnt_d', 'last_credit_pull_d', 'application_type', 'hardship_flag', 'disbursement_method', 'debt_settlement_flag']

Missing values in df: 2,439,863


In [8]:
# Numerical: median from train
num_medians = train[num_cols].median().fillna(0)

# Categorical: mode from train
cat_modes = {}
for col in cat_cols:
    if col in train.columns:
        mode_series = train[col].mode()
        cat_modes[col] = mode_series[0] if len(mode_series) > 0 else 'Unknown'

# Apply to df (needed for encoding cell), train, val, test
for target_df in [df, train, val, test]:
    # Numerical
    for col in num_cols:
        if col in target_df.columns:
            target_df[col] = target_df[col].fillna(num_medians[col])
    # Categorical
    for col, mode_val in cat_modes.items():
        if col in target_df.columns:
            target_df[col] = target_df[col].fillna(mode_val)

print(f"Missing in df: {df.isnull().sum().sum()}")
print(f"Missing — Train: {train.isnull().sum().sum()}, "
      f"Val: {val.isnull().sum().sum()}, "
      f"Test: {test.isnull().sum().sum()}")


Missing in df: 0
Missing — Train: 0, Val: 0, Test: 0


In [9]:
# Drop high-cardinality date/location columns
drop_high_card = ['earliest_cr_line', 'last_pymnt_d', 'last_credit_pull_d', 'addr_state']
df = df.drop(columns=[c for c in drop_high_card if c in df.columns])

# Ordinal encodings
grade_order = {'A':1,'B':2,'C':3,'D':4,'E':5,'F':6,'G':7}
df['grade'] = df['grade'].map(grade_order)

sub_grades = [f'{g}{n}' for g in 'ABCDEFG' for n in range(1, 6)]
df['sub_grade'] = df['sub_grade'].map({v: i+1 for i, v in enumerate(sub_grades)})

emp_order = {'< 1 year':0,'1 year':1,'2 years':2,'3 years':3,'4 years':4,
             '5 years':5,'6 years':6,'7 years':7,'8 years':8,'9 years':9,'10+ years':10}
df['emp_length'] = df['emp_length'].map(emp_order)

df['term'] = df['term'].str.strip().map({'36 months': 36, '60 months': 60})

# One-hot encode remaining categorical columns
# NOTE: debt_settlement_flag included here → becomes debt_settlement_flag_Y

cat_to_encode = [c for c in df.select_dtypes(include='object').columns if c != 'target']
df = pd.get_dummies(df, columns=cat_to_encode, drop_first=True)

print(f"Encoded df shape: {df.shape}")
print(f"Missing in encoded df: {df.isnull().sum().sum()}")

Encoded df shape: (1329272, 96)
Missing in encoded df: 0


In [11]:
feature_cols = [c for c in df.columns if c != 'target']

# Re-split using saved indices
train_enc = df[df.index.isin(train.index)]
val_enc   = df[df.index.isin(val.index)]
test_enc  = df[df.index.isin(test.index)]

X_train, y_train = train_enc[feature_cols], train_enc['target']
X_val,   y_val   = val_enc[feature_cols],   val_enc['target']
X_test,  y_test  = test_enc[feature_cols],  test_enc['target']

# Scale: fit on train ONLY
scaler  = MinMaxScaler()
X_train = pd.DataFrame(scaler.fit_transform(X_train), columns=feature_cols)
X_val   = pd.DataFrame(scaler.transform(X_val),       columns=feature_cols)
X_test  = pd.DataFrame(scaler.transform(X_test),      columns=feature_cols)

print(f"X_train: {X_train.shape} | default rate: {y_train.mean():.3f}")
print(f"X_val:   {X_val.shape}   | default rate: {y_val.mean():.3f}")
print(f"X_test:  {X_test.shape}  | default rate: {y_test.mean():.3f}")

X_train: (449858, 95) | default rate: 0.171
X_val:   (375220, 95)   | default rate: 0.205
X_test:  (504194, 95)  | default rate: 0.265


In [12]:
# 1. Post-origination outcome columns (known only AFTER loan completes)
outcome_leakage = [
    'total_pymnt', 'total_pymnt_inv', 'total_rec_prncp', 'total_rec_int',
    'total_rec_late_fee', 'recoveries', 'collection_recovery_fee',
    'last_pymnt_amnt', 'out_prncp', 'out_prncp_inv',
    'funded_amnt_inv', 'installment'
]

# 2. Post-origination flag 
flag_leakage = ['debt_settlement_flag_Y']

all_leakage = outcome_leakage + flag_leakage
found = [c for c in all_leakage if c in X_train.columns]
print(f"Dropping {len(found)} leakage columns: {found}")

for df_ in [X_train, X_val, X_test]:
    df_.drop(columns=found, inplace=True, errors='ignore')

print(f"Final feature count: {X_train.shape[1]}")  # expect 82
print(f"debt_settlement_flag_Y present: {'debt_settlement_flag_Y' in X_train.columns}")

Dropping 13 leakage columns: ['total_pymnt', 'total_pymnt_inv', 'total_rec_prncp', 'total_rec_int', 'total_rec_late_fee', 'recoveries', 'collection_recovery_fee', 'last_pymnt_amnt', 'out_prncp', 'out_prncp_inv', 'funded_amnt_inv', 'installment', 'debt_settlement_flag_Y']
Final feature count: 82
debt_settlement_flag_Y present: False


In [13]:
os.makedirs('acrs_data', exist_ok=True)

X_train.to_parquet('acrs_data/X_train.parquet')
X_val.to_parquet('acrs_data/X_val.parquet')
X_test.to_parquet('acrs_data/X_test.parquet')

np.save('acrs_data/y_train.npy', y_train.values)
np.save('acrs_data/y_val.npy',   y_val.values)
np.save('acrs_data/y_test.npy',  y_test.values)

print("All files saved to acrs_data/")
print(f"  X_train: {X_train.shape}")
print(f"  X_val:   {X_val.shape}")
print(f"  X_test:  {X_test.shape}")

All files saved to acrs_data/
  X_train: (449858, 82)
  X_val:   (375220, 82)
  X_test:  (504194, 82)


In [25]:
print("=" * 50)
print("PHASE 1 — DATA PIPELINE VERIFICATION")
print("=" * 50)
print(f"Train rows:    {X_train.shape[0]:,}")
print(f"Val rows:      {X_val.shape[0]:,}")
print(f"Test rows:     {X_test.shape[0]:,}")
print(f"Features:      {X_train.shape[1]}")
print(f"Missing train: {X_train.isnull().sum().sum()}")
print(f"Missing val:   {X_val.isnull().sum().sum()}")
print(f"Missing test:  {X_test.isnull().sum().sum()}")
print("=" * 50)
print("PHASE 1 COMPLETE — Ready for modelling")

PHASE 1 — DATA PIPELINE VERIFICATION
Train rows:    449,858
Val rows:      375,220
Test rows:     504,194
Features:      82
Missing train: 0
Missing val:   0
Missing test:  0
PHASE 1 COMPLETE — Ready for modelling
